# Proyecto Final - Adult Income
## Guía general del proyecto

Este notebook sustituye como guía organizada al borrador exploratorio `Proyecto_Final_Mod6.ipynb`. Resume el flujo completo y enlaza conceptualmente los notebooks especializados sin duplicar su código.

## Problema y objetivo
El proyecto busca estimar si el ingreso anual de una persona supera USD 50K utilizando información demográfica y laboral del dataset Adult. El objetivo de modelado es una clasificación binaria; el análisis debe reconocer el desbalance y los posibles riesgos al trabajar con atributos sensibles.

## Orden recomendado de ejecución

1. `00_ingest.ipynb`: descarga reproducible y almacenamiento raw.
2. `01_data_quality_eda.ipynb`: siete gates y EDA asociado a decisiones.
3. `02_feature_engineering.ipynb`: transformación compartida y control de leakage.
4. `03_pipeline_adult.ipynb`: entrenamiento y persistencia.
5. `04_evaluar_modelo.ipynb`: evaluación detallada del modelo guardado.

La ingesta puede omitirse cuando `adult_raw.csv` ya está disponible.

## Arquitectura

```text
UCI -> raw CSV -> normalización -> quality gates -> split estratificado
                                              -> features + modelos candidatos -> validación -> modelo ganador -> evaluación
```

Los notebooks explican la arquitectura. Los scripts `.py` permiten automatizarla y los módulos `src/` constituyen la única fuente de lógica reutilizable.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import load_adult
df = load_adult()
print(f'Dataset disponible: {df.shape[0]:,} filas y {df.shape[1]} columnas')
df.head()

## G. Calidad automática
El pipeline revisa esquema, volumen, duplicados, completitud y dominio del target, faltantes de features y rangos numéricos. El entrenamiento no continúa silenciosamente con datos fuera del contrato.

In [ ]:
from src.quality import run_quality_gates
quality_report = run_quality_gates(df)
quality_report

## H. EDA orientado a decisiones
Cada análisis debe contestar qué cambia. En este proyecto el balance cambia las métricas y pesos; los faltantes cambian la estrategia de imputación; los ceros estructurales no se consideran nulos; y las relaciones no lineales apoyan el uso de árboles.

## I. Feature Engineering reutilizable
La función `build_feature_engineering()` se usa tanto en notebooks como en producción. Las estadísticas de imputación y codificación se ajustan solo con train y quedan incorporadas en el modelo serializado.

In [ ]:
from src.modeling import build_candidate_pipelines
build_candidate_pipelines()

## Resultados verificados
La comparación de referencia evaluó regresión logística, árbol de decisión, KNN y Random Forest. La regresión logística obtuvo el mayor ROC-AUC de validación (0.906) y fue seleccionada. En test obtuvo aproximadamente accuracy 0.807, F1 0.675, ROC-AUC 0.904 y Average Precision 0.757. Estos resultados deben reproducirse con la misma versión de datos, dependencias y semilla.

## Alcance
Esta organización documenta ingesta, calidad, EDA, features, entrenamiento y evaluación. MLflow, API, Docker, monitoreo, drift y CI/CD pertenecen a fases posteriores de la guía y no deben declararse terminados hasta implementarlos y probarlos.